# File

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("ML_Ready_Extended_Geography.csv") # ETO YUNG GINAMIT KONG CSV YA CHINECK Q NA
X = df.drop('HIV_Status', axis=1)
y = df['HIV_Status']

# Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
df.head(10)

,Sex,Age_Group,Transmission,Healthcare_Access_Friction,Civil_Status,OFW_Status,Chemsex_Engagement,Alcohol_Sex_Risk,PrEP_Awareness,Transactional_Sex,STI_BBV_CoInfection_Any,HIV_Status
0,Female,<15,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
1,Male,15-24,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
2,Male,15-24,Male to Male Sex,2,Single,No,No,No,Yes,No TS,Yes,Reactive
3,Male,15-24,Male to Male Sex,2,Single,No,No,No,No,No TS,Yes,Reactive
4,Male,15-24,Male to Male Sex,2,Single,No,Yes,No,Yes,No TS,No,Reactive
5,Male,15-24,Male to Male/Female Sex,2,Single,No,No,Yes,Yes,Both,No,Reactive
6,Male,25-34,Male to Female Sex,2,Common-Law,No,No,No,No,No TS,No,Reactive
7,Male,25-34,Male to Male Sex,2,Single,No,No,No,Yes,No TS,No,Reactive
8,Male,25-34,Male to Male Sex,2,Single,No,No,No,No,No TS,No,Reactive
9,Male,25-34,Male to Male Sex,2,Single,No,No,Yes,Yes,Paid for sex,Yes,Reactive


# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# list num and cat
numeric_features = []
categorical_features = ['Sex','Age_Group','Transmission',
                        'Healthcare_Access_Friction','Civil_Status','OFW_Status','Chemsex_Engagement',
                        'Alcohol_Sex_Risk','PrEP_Awareness','Transactional_Sex','STI_BBV_CoInfection_Any']

# 2. proprocessor
preprocessor = ColumnTransformer(
    transformers=[
        # Apply StandardScaler to numeric columns
        ('num', StandardScaler(), numeric_features),

        # Apply OneHotEncoder to categorical columns
        # handle_unknown='ignore' ensures the code won't crash if X_test has a category not seen in X_train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not listed above (or use 'drop')
)

# 3. Fit and Transform X_train
# CRITICAL: We fit ONLY on X_train to learn the mean/std and categories.
# Then we transform both train and test using those learned values.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# convert into df
X# Convert into DataFrame AND preserve the original index
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

# Dealing with y columns
mapping = {'Non-Reactive': 0, 'Reactive': 1}

# y_train and y_test already keep their indices, but it's good practice
# to ensure it's a Series or 1D array for Scikit-Learn classifiers
y_train_processed = y_train.map(mapping)
y_test_processed = y_test.map(mapping)

 # MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, average_precision_score

# 1. Define the base model
# We keep class_weight='balanced' here so every variation uses it
rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1 # Uses all your CPU cores to speed up the search
)

# 2. Define the hyperparameter grid
# We provide a few options for the model to test out
param_grid = {
    'n_estimators': [100, 200],           # Number of trees
    'max_depth': [None, 10, 20],          # Maximum depth of the trees (prevents overfitting)
    'min_samples_split': [2, 5, 10],      # Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4]         # Minimum samples required at a leaf node
}

# 3. Initialize GridSearchCV
# CRITICAL: We set scoring='average_precision' so it optimizes for the minority class!
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='average_precision', # Alternatively, you can use scoring='f1'
    cv=5,                        # 5-fold cross-validation
    verbose=2,                   # Prints out progress so you aren't staring at a blank screen
    n_jobs=-1                    # Speeds up the cross-validation across CPU cores
)

# 4. Fit the Grid Search on your processed training data
print("Starting Grid Search... this might take a minute! ☕")
grid_search.fit(X_train_processed, y_train_processed)

# 5. Extract the best model
best_rf_model = grid_search.best_estimator_

print("\n--- Grid Search Results ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation AUPRC: {grid_search.best_score_:.4f}")

# 6. Make predictions using the winning model
y_pred = best_rf_model.predict(X_test_processed)
y_probs = best_rf_model.predict_proba(X_test_processed)[:, 1]

# 7. Final Evaluation
print("\n--- Final Classification Report ---")
print(classification_report(y_test_processed, y_pred, target_names=['Non Reactive', 'Reactive']))
print(f"Test F1-Score: {f1_score(y_test_processed, y_pred):.4f}")
print(f"Final AUPRC: {average_precision_score(y_test_processed, y_probs):.4f}")

# Threshold Moving 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,
    average_precision_score
)

# ---------------------------------------------------------
# STEP 1: FIND THE BEST THRESHOLD ON THE *VALIDATION* SET
# ---------------------------------------------------------

# Get probabilities for the Validation set
rf_probs_val = best_rf_model.predict_proba(X_val_processed)[:, 1]

# Vectorized calculation of all thresholds on the validation set
precisions_val, recalls_val, thresholds_val = precision_recall_curve(y_val_processed, rf_probs_val)

# Calculate F1 mathematically (1e-8 prevents division by zero)
f1_scores_val = 2 * (precisions_val * recalls_val) / (precisions_val + recalls_val + 1e-8)

# Find the exact threshold that maximizes F1 on validation data
best_idx = np.argmax(f1_scores_val[:-1])
best_threshold = thresholds_val[best_idx]
best_f1_val = f1_scores_val[best_idx]

print(f"Optimal Threshold (from Val Set): {best_threshold:.2f} (or {best_threshold*100:.0f}%)")

# ---------------------------------------------------------
# STEP 2: EVALUATE THIS THRESHOLD ON THE *TEST* SET
# ---------------------------------------------------------

# Get probabilities for the unseen Test set
rf_probs_test = best_rf_model.predict_proba(X_test_processed)[:, 1]

# Apply the optimal threshold
optimal_preds_test = (rf_probs_test >= best_threshold).astype(int)
default_preds_test = best_rf_model.predict(X_test_processed) # Default 0.5 threshold

print("\n--- FINAL TEST SET PERFORMANCE ---")
print(f"Default (50%) F1-Score: {f1_score(y_test_processed, default_preds_test):.4f}")
print(f"New Maximum F1-Score:   {f1_score(y_test_processed, optimal_preds_test):.4f}")
print(f"New Precision:          {precision_score(y_test_processed, optimal_preds_test, zero_division=0):.4f}")
print(f"New Recall:             {recall_score(y_test_processed, optimal_preds_test):.4f}\n")

In [ ]:

# ---------------------------------------------------------
# STEP 3: VISUALIZE THE RESULTS
# ---------------------------------------------------------

# Calculate Precision-Recall Curve data for the Test set
precision_rf, recall_rf, pr_thresholds_rf = precision_recall_curve(y_test_processed, rf_probs_test)
auprc_rf = average_precision_score(y_test_processed, rf_probs_test)

# Calculate Test F1 scores for plotting
f1_rf = 2 * (precision_rf * recall_rf) / (precision_rf + recall_rf + 1e-8)

# Create the Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# --- Plot 1: Metrics vs. Threshold ---
ax1.plot(pr_thresholds_rf, precision_rf[:-1], label='Precision', color='blue', linestyle='--')
ax1.plot(pr_thresholds_rf, recall_rf[:-1], label='Recall', color='green', linestyle='--')
ax1.plot(pr_thresholds_rf, f1_rf[:-1], label='F1-Score', color='red', linewidth=2)
ax1.axvline(x=best_threshold, color='black', linestyle=':', label=f'Optimal Threshold ({best_threshold:.2f})')

ax1.set_title("Metrics vs. Decision Threshold (Test Data)", fontsize=14)
ax1.set_xlabel("Decision Threshold", fontsize=12)
ax1.set_ylabel("Score", fontsize=12)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.legend(loc='lower center')
ax1.grid(alpha=0.3)

# --- Plot 2: Precision-Recall Curve ---
ax2.plot(recall_rf, precision_rf, color='purple', linewidth=2, label=f'RF PR Curve (AUPRC = {auprc_rf:.3f})')

# Mark the spot on the PR curve where our best threshold lands
optimal_recall = recall_score(y_test_processed, optimal_preds_test)
optimal_precision = precision_score(y_test_processed, optimal_preds_test, zero_division=0)
ax2.scatter(optimal_recall, optimal_precision, color='red', s=100, zorder=5, label='Optimal Operating Point')

ax2.set_title("Precision-Recall Curve (Test Data)", fontsize=14)
ax2.set_xlabel("Recall", fontsize=12)
ax2.set_ylabel("Precision", fontsize=12)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.legend(loc='lower left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# SHAP

In [ ]:
import shap

# 1. Get the raw SHAP values from your winning Random Forest
explainer = shap.TreeExplainer(best_rf_model)
shap_vals_output = explainer.shap_values(X_test_processed)

# Check if SHAP gave us a list or a 3D array, and extract Class 1 (Reactive) safely
if isinstance(shap_vals_output, list):
    shap_raw = shap_vals_output[1]
elif len(shap_vals_output.shape) == 3:
    # Slice the 3D array: [All Rows, All Columns, Class 1]
    shap_raw = shap_vals_output[:, :, 1]
else:
    shap_raw = shap_vals_output
# -----------------------

# Put them in a DataFrame so we can easily group the columns by name
shap_df = pd.DataFrame(shap_raw, columns=X_test_processed.columns)
grouped_shap_df = shap_df.copy()

# 2. Define the exact prefixes of your One-Hot Encoded columns
categorical_prefixes = ['cat__Sex','cat__Age_Group','cat__Transmission',
                        'cat__Healthcare_Access_Friction','cat__Civil_Status','cat__OFW_Status','cat__Chemsex_Engagement',
                        'cat__Alcohol_Sex_Risk','cat__PrEP_Awareness','cat__Transactional_Sex','cat__STI_BBV_CoInfection_Any']

# 3.Summing the SHAP values
for prefix in categorical_prefixes:
    # Find all dummy columns that start with this prefix
    dummy_cols = [col for col in X_test_processed.columns if col.startswith(f"{prefix}_")]

    if len(dummy_cols) > 0:
        # The true SHAP value of the original category is the SUM of its dummy parts
        grouped_shap_df[prefix] = shap_df[dummy_cols].sum(axis=1)

        # Drop the diluted dummy columns so they don't clutter the plot
        grouped_shap_df.drop(columns=dummy_cols, inplace=True)


In [ ]:
# 4. Plot the Global Feature Importance (Miku Teal!)
plt.figure(figsize=(10, 6))
plt.title("True Feature Importance (Categorical Variables Grouped)", fontsize=14)

shap.summary_plot(
    grouped_shap_df.values,
    feature_names=grouped_shap_df.columns,
    plot_type="bar",          # Bar plots are best for grouped categorical SHAP!
    color="#39C5BB",          # Miku color to match your dashboard
    show=False
)

# Optional: Make it look extra clean
plt.grid(axis='x', alpha=0.3)
plt.show()

## TOP 10 INDIVIDUAL FEATURES 

In [ ]:
import shap
import matplotlib.pyplot as plt

# 1. Calculate SHAP values 
explainer = shap.TreeExplainer(best_rf_model)
shap_vals_output = explainer.shap_values(X_test_processed)

# 2. Safely extract Class 1 (Reactive)
if isinstance(shap_vals_output, list):
    shap_raw = shap_vals_output[1]
elif len(shap_vals_output.shape) == 3:
    shap_raw = shap_vals_output[:, :, 1]
else:
    shap_raw = shap_vals_output

# 3. Plot the Top 10 Features (Miku Teal Bar Chart)
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_raw,                                # Use the safely extracted values
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="bar",                         # Bar chart
    color="#39C5BB",
    max_display=10,
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

## Beeswarm

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# 1.(Extract Class 1 safely so it doesn't crash)
if isinstance(shap_vals_output, list):
    shap_raw = shap_vals_output[1]
elif len(shap_vals_output.shape) == 3:
    shap_raw = shap_vals_output[:, :, 1]
else:
    shap_raw = shap_vals_output

# 2. The Plot
plt.figure(figsize=(10, 6))
miku_cmap = mcolors.LinearSegmentedColormap.from_list("miku_gradient", ["#D1F2F0", "#39C5BB"])

shap.summary_plot(
    shap_raw,                                # <-- Use the safely extracted array!
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="dot",                         # Beeswarm style
    cmap=miku_cmap,
    max_display=10,                          # Top 10 only
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

# SAVING THE MODEL (RUN MO TO YA IF YOU THINK MAGANDA RESULT PARA MASAVE)

In [ ]:
import joblib
model_filename = 'best_rf_model.joblib'
joblib.dump(best_rf_model, model_filename)

print(f"\nModel successfully saved to {model_filename}")